<a href="https://colab.research.google.com/github/gabriel-tfg/Practica2_ARP/blob/main/SB3_cartpole_500pasos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gymnasium stable_baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 6.4 MB/s eta 0:00:00


Entrenamiento de la política con SB3. Al activar el flag de verbose, observamos que SB3 nos ofrece estadísticas tales como reward total, número de pasos hasta cambiar de episodio, etcétera. Acordémonos que el CartPole se resetea automáticamente a los 500 episodios o bien si el péndulo se cae.



In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

env = gym.make("CartPole-v1", render_mode="rgb_array")

model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    gamma=0.99,
    gae_lambda=0.95,
    ent_coef=0.0
)

model.learn(total_timesteps=100000)

mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)
print("Recompensa media:", mean_reward)
print("Desviación típica:", std_reward)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 23.5     |
|    ep_rew_mean     | 23.5     |
| time/              |          |
|    fps             | 541      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 27.4         |
|    ep_rew_mean          | 27.4         |
| time/                   |              |
|    fps                  | 481          |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0082291225 |
|    clip_fraction        | 0.0989       |
|    clip_range           | 0.2          |
|    entropy_loss         | -0.686       |
|    explained_variance   | -0.00466     |
|    learning_r

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Recompensa media: 500.0
Desviación típica: 0.0


In [ ]:
# Evaluación cuantitativa
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=20, deterministic=True)

print("Recompensa media:", mean_reward)
print("Desviación típica:", std_reward)

Recompensa media: 500.0
Desviación típica: 0.0


In [ ]:
# Ver episodios individuales
for ep in range(10):
    obs, info = env.reset(seed=42 + ep)
    done = False
    truncated = False
    total_reward = 0

    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_reward += reward

    print(f"Episodio {ep+1}: {total_reward}")

Episodio 1: 500.0
Episodio 2: 500.0
Episodio 3: 500.0
Episodio 4: 500.0
Episodio 5: 500.0
Episodio 6: 500.0
Episodio 7: 500.0
Episodio 8: 500.0
Episodio 9: 500.0
Episodio 10: 500.0


Probamos la política en el entorno

In [ ]:
# Listado de cuadros para guardar el video
frames = []

# Reset del entorno
observation, info = env.reset(seed=42)

for _ in range(1000):
    frame = env.render()
    frames.append(frame)

    action, _ = model.predict(observation, deterministic=True)

    observation, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        observation, info = env.reset()

# Cerrar el entorno
env.close()

Guardamos el vídeo

In [ ]:
# Crear el video a partir de los cuadros guardados
video_filename = "cartpole_dqn_policy.mp4"
height, width, _ = frames[0].shape  # Obtener dimensiones de los cuadros

# Configuración de salida para formato MP4
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video = cv2.VideoWriter(video_filename, fourcc, 30.0, (width, height))

# Escribir cada cuadro en el video
for frame in frames:
    video.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))  # Convertir de RGB a BGR para OpenCV

# Liberar el objeto VideoWriter
video.release()

print(f"Video guardado como {video_filename}")


Video guardado como cartpole_dqn_policy.mp4


Visualizamos el vídeo

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# Input video path
save_path = "cartpole_dqn_policy.mp4"

# Compressed video path
compressed_path = "result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")
# Show video
mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=800 controls>
      <source src="%s" type="video/mp4">
</video>""" % data_url)

¿Qué observas con respecto al CartPole aleatorio?